In [1]:
# usage: python run_md.py <start.pdb> <out_dir> [--prod_ns 200] [--save_ps 1] [--seed 0]
import ctypes, glob, sys
_lib = glob.glob(f"{sys.prefix}/lib/python3.13/site-packages/nvidia/**/libnvrtc-builtins.so.13*", recursive=True)
assert _lib, "libnvrtc-builtins not found in the venv. Reinstall nvidia-cuda-nvrtc"
ctypes.CDLL(_lib[0], mode=ctypes.RTLD_GLOBAL)

import argparse, os
import openmm as mm, openmm.app as app, openmm.unit as u

args = {
    'out_dir': 'robin_md_trajectories',
    'prod_ns': 5,
    'save_ps': 0.01,
    'seed': 0.,
    'platform': 'CUDA',
}

In [2]:
import openmm; print([openmm.Platform.getPlatform(i).getName() for i in range(openmm.Platform.getNumPlatforms())])

['Reference', 'CPU', 'CUDA', 'OpenCL']


In [ ]:
from tqdm import tqdm
import os, glob
import multiprocessing as mp
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

from run_md import run_sim, init_worker
NVRTC_DIR = _lib[0]
GPUS = [0, 1, 2, 3, 4]
JOBS_PER_GPU = 2        # 33–63 atoms barely load a GPU; benchmark 1, 2, 4

# Set in the parent BEFORE the pool starts: spawned workers inherit it at process
# start, which is when the loader reads it. That fixes the NVRTC error for workers.
os.environ["LD_LIBRARY_PATH"] = f"{NVRTC_DIR}:{os.environ.get('LD_LIBRARY_PATH', '')}"

ctx = mp.get_context("spawn")                 # never fork with CUDA
gpu_q = ctx.Queue()
for g in GPUS:
    for _ in range(JOBS_PER_GPU):
        gpu_q.put(g)                         # one token per worker slot

pdbs = sorted(glob.glob("robin_samples/*/*.pdb"))
with ProcessPoolExecutor(max_workers=len(GPUS) * JOBS_PER_GPU, mp_context=ctx,
                         initializer=init_worker, initargs=(gpu_q,)) as ex:
    futures = {ex.submit(run_sim, p): p
               for i, p in enumerate(pdbs)}
    failed = []
    with tqdm(total=len(futures), desc="MD runs", unit="run") as bar:
        for f in as_completed(futures):
            pdb = futures[f]
            try:
                f.result()
            except Exception as e:
                failed.append((pdb, repr(e)))
                tqdm.write(f"FAILED: {Path(pdb).name}: {e!r}")   # prints without breaking the bar
            bar.set_postfix(failed=len(failed))
            bar.update(1)

KeyboardInterrupt: 

: 